In [5]:
import pandas as pd
import numpy as np
import os
import sqlite3

from sqlalchemy import create_engine



In [26]:
df = pd.read_csv(
    "../data/raw/02_nav_history.csv"
)

In [27]:
df.head()

,date,nav,amfi_code
0,19-06-2026,105.9219,119551
1,18-06-2026,105.9003,119551
2,17-06-2026,105.8482,119551
3,16-06-2026,105.8455,119551
4,15-06-2026,105.8125,119551


In [28]:
nav.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB


In [30]:
nav['date'] = pd.to_datetime(
    nav['date']
)

In [31]:
nav.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB


In [32]:
nav = nav.sort_values(
    ['amfi_code', 'date']
)

In [33]:
nav = nav.reset_index(drop=True)

In [34]:
nav[nav['nav'] <= 0]

,amfi_code,date,nav


In [39]:
nav['nav'] = (
    nav.groupby('amfi_code')['nav']
    .ffill()
)

In [38]:
nav['nav'].isnull().sum()

np.int64(0)

In [40]:
nav.head()


,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [41]:
nav.tail()

,amfi_code,date,nav
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873
45999,149324,2026-05-29,279.7511


In [43]:
import os

os.makedirs(
    "../data/processed",
    exist_ok=True
)

In [44]:
nav.to_csv(
    "../data/processed/02_nav_history_clean.csv",
    index=False
)

In [45]:
os.listdir("../data/processed")

['02_nav_history_clean.csv']

In [46]:
transactions = pd.read_csv(
    "../data/raw/08_investor_transactions.csv"
)

In [47]:
transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [48]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [49]:
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date'],
    format='%Y-%m-%d'
)

In [50]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

In [51]:
transactions['transaction_type'].unique()

array(['SIP', 'Redemption', 'Lumpsum'], dtype=object)

In [52]:
transactions['transaction_type'] = (
    transactions['transaction_type']
    .str.upper()
    .str.strip()
)

In [53]:
transactions['transaction_type'].unique()

array(['SIP', 'REDEMPTION', 'LUMPSUM'], dtype=object)

In [54]:
transactions[
    transactions['amount_inr'] <= 0
]

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status


In [55]:
transactions['kyc_status'].unique()

array(['Verified', 'Pending'], dtype=object)

In [56]:
valid_kyc = [
    'Verified',
    'Pending',
    'Rejected'
]

transactions[
    ~transactions['kyc_status'].isin(valid_kyc)
]

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status


In [57]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

In [58]:
transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,REDEMPTION,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,LUMPSUM,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [60]:
transactions.to_csv(
    "../data/processed/08_investor_transactions_clean.csv",
    index=False
)

In [61]:
import os

os.listdir("../data/processed")

['02_nav_history_clean.csv', '08_investor_transactions_clean.csv']

In [65]:
performance = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

In [66]:
performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [67]:
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [68]:
performance.isnull().sum()

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

In [69]:
expense_anomalies = performance[
    (performance['expense_ratio_pct'] < 0.1) |
    (performance['expense_ratio_pct'] > 2.5)
]

expense_anomalies

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [70]:
performance[
    (performance['return_5yr_pct'] > 100) |
    (performance['return_5yr_pct'] < -100)
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [71]:
performance.to_csv(
    "../data/processed/07_scheme_performance_clean.csv",
    index=False
)

In [72]:
import os

os.listdir("../data/processed")

['02_nav_history_clean.csv',
 '07_scheme_performance_clean.csv',
 '08_investor_transactions_clean.csv']

In [15]:
import sqlite3

conn = sqlite3.connect(
    "../bluestock_mf.db"
)

cur = conn.cursor()

In [16]:
with open("../sql/schema.sql","r") as file:
    schema = file.read()

cur.executescript(schema)

conn.commit()

OperationalError: table dim_fund already exists

In [19]:
pd.read_sql(
    """
    SELECT name 
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

,name
0,dim_fund
1,dim_date
2,fact_nav
3,sqlite_sequence
4,fact_transactions
5,fact_performance
6,fact_aum


In [20]:
engine = create_engine(
    "sqlite:///../bluestock_mf.db"
)

In [21]:
fund = pd.read_csv(
    "../data/raw/01_fund_master.csv"
)

In [22]:
fund.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [23]:
fund.to_sql(
    "dim_fund",
    engine,
    if_exists="append",
    index=False
)

40

In [24]:
nav = pd.read_csv(
    "../data/processed/02_nav_history_clean.csv"
)

In [25]:
nav['date'] = pd.to_datetime(
    nav['date']
)

In [26]:
nav.to_sql(
    "fact_nav",
    engine,
    if_exists="append",
    index=False
)

46000

In [27]:
transactions = pd.read_csv(
    "../data/processed/08_investor_transactions_clean.csv"
)

In [29]:
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date']
)
transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="append",
    index=False
)

32778

In [32]:
performance = pd.read_csv(
    "../data/processed/07_scheme_performance_clean.csv"
)

In [33]:
performance.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

OperationalError: (sqlite3.OperationalError) table fact_performance has no column named scheme_name
[SQL: INSERT INTO fact_performance (amfi_code, scheme_name, fund_house, category, "plan", return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: [(119551, 'SBI Bluechip Fund - Regular Plan - Growth', 'SBI Mutual Fund', 'Large Cap', 'Regular', 12.42, 12.36, 14.45, 11.49, 0.87, 0.89, 0.88, 1.29, 14.0, -21.7, 14288, 1.54, 4, 'Moderate'), (119552, 'SBI Bluechip Fund - Direct Plan - Growth', 'SBI Mutual Fund', 'Large Cap', 'Direct', 15.25, 11.3, 14.23, 9.52, 1.78, 0.87, 0.81, 1.29, 14.0, -24.43, 1231, 0.66, 3, 'Moderate'), (119598, 'SBI Small Cap Fund - Regular Plan - Growth', 'SBI Mutual Fund', 'Small Cap', 'Regular', 24.56, 23.39, 20.67, 22.16, 1.23, 0.89, 0.94, 1.

In [34]:
conn = sqlite3.connect("../bluestock_mf.db")

cursor = conn.cursor()

cursor.execute("""
DROP TABLE IF EXISTS fact_performance;
""")

conn.commit()

In [35]:
cursor.execute("""

CREATE TABLE fact_performance (

    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,

    amfi_code INTEGER,

    scheme_name TEXT,

    fund_house TEXT,

    category TEXT,

    plan TEXT,

    return_1yr_pct REAL,

    return_3yr_pct REAL,

    return_5yr_pct REAL,

    benchmark_3yr_pct REAL,

    alpha REAL,

    beta REAL,

    sharpe_ratio REAL,

    sortino_ratio REAL,

    std_dev_ann_pct REAL,

    max_drawdown_pct REAL,

    aum_crore REAL,

    expense_ratio_pct REAL,

    morningstar_rating INTEGER,

    risk_grade TEXT,

    FOREIGN KEY(amfi_code)
    REFERENCES dim_fund(amfi_code)

);

""")

conn.commit()

In [36]:
performance.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

40

In [37]:
pd.read_sql(
"""
SELECT COUNT(*) AS rows
FROM fact_performance;
""",
conn
)

,rows
0,40


In [38]:
pd.read_sql(
"""
SELECT COUNT(*) AS rows
FROM fact_performance;
""",
conn
)

,rows
0,40


In [39]:
pd.read_sql(
"""
SELECT COUNT(*) AS rows
FROM fact_nav;
""",
conn
)

,rows
0,46000


In [40]:
pd.read_sql(
"""
SELECT COUNT(*) AS rows
FROM fact_transactions;
""",
conn
)

,rows
0,32778


In [41]:
aum = pd.read_csv(
    "../data/raw/03_aum_by_fund_house.csv"
)

In [42]:
aum.head()

,date,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000,186
1,2022-03-31,ICICI Prudential MF,4.65,465000,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195
3,2022-03-31,Nippon India MF,2.70,270000,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168


In [43]:
aum['date'] = pd.to_datetime(
    aum['date']
)

In [45]:
aum.to_sql(
    "fact_aum",
    engine,
    if_exists="append",
    index=False
)

90

In [46]:
pd.read_sql(
"""
SELECT COUNT(*) AS rows
FROM fact_aum;
""",
conn
)

,rows
0,90


In [47]:
pd.read_sql(
"""
SELECT 
name
FROM sqlite_master
WHERE type='table';
""",
conn
)

,name
0,dim_fund
1,dim_date
2,fact_nav
3,sqlite_sequence
4,fact_transactions
5,fact_aum
6,fact_performance


In [48]:
pd.read_sql(
"""
SELECT *
FROM fact_nav
LIMIT 5;
""",
conn
)

,nav_id,amfi_code,date,nav
0,1,100016,2022-01-03 00:00:00.000000,520.4608
1,2,100016,2022-01-04 00:00:00.000000,515.0971
2,3,100016,2022-01-05 00:00:00.000000,521.7239
3,4,100016,2022-01-06 00:00:00.000000,515.7880
4,5,100016,2022-01-07 00:00:00.000000,515.1639


In [49]:
pd.read_sql(
"""
SELECT *
FROM fact_transactions
LIMIT 5;
""",
conn
)

,transaction_id,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,1,INV003054,2024-01-01 00:00:00.000000,119092,SIP,1834.0,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,2,INV002952,2024-01-01 00:00:00.000000,148567,REDEMPTION,392882.0,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,3,INV003420,2024-01-01 00:00:00.000000,118636,SIP,912.0,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,4,INV003436,2024-01-01 00:00:00.000000,118634,SIP,1102.0,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,5,INV004691,2024-01-01 00:00:00.000000,119094,LUMPSUM,8682.0,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [50]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(
    "../bluestock_mf.db"
)

In [51]:
query = """

SELECT 
scheme_name,
return_5yr_pct
FROM fact_performance
ORDER BY return_5yr_pct DESC
LIMIT 5;

"""

pd.read_sql(query, conn)

,scheme_name,return_5yr_pct
0,ABSL Small Cap Fund - Regular - Growth,23.80
1,Axis Small Cap Fund - Regular - Growth,22.62
2,Nippon India Small Cap Fund - Regular - Growth,21.88
3,SBI Small Cap Fund - Direct Plan - Growth,21.82
4,SBI Small Cap Fund - Regular Plan - Growth,20.67


In [52]:
import os
import pandas as pd

raw_path = "../data/raw"
processed_path = "../data/processed"

os.makedirs(processed_path, exist_ok=True)


def save_clean_copy(filename):
    df = pd.read_csv(
        f"{raw_path}/{filename}.csv"
    )

    df.drop_duplicates(inplace=True)

    df.to_csv(
        f"{processed_path}/{filename}_clean.csv",
        index=False
    )

    print(filename, "completed", df.shape)

In [53]:
save_clean_copy("01_fund_master")
save_clean_copy("03_aum_by_fund_house")
save_clean_copy("04_monthly_sip_inflows")
save_clean_copy("05_category_inflows")
save_clean_copy("06_industry_folio_count")
save_clean_copy("09_portfolio_holdings")
save_clean_copy("10_benchmark_indices")

01_fund_master completed (40, 15)
03_aum_by_fund_house completed (90, 5)
04_monthly_sip_inflows completed (48, 6)
05_category_inflows completed (144, 3)
06_industry_folio_count completed (21, 6)
09_portfolio_holdings completed (322, 8)
10_benchmark_indices completed (8050, 3)


In [54]:
os.listdir("../data/processed")

['01_fund_master_clean.csv',
 '02_nav_history_clean.csv',
 '03_aum_by_fund_house_clean.csv',
 '04_monthly_sip_inflows_clean.csv',
 '05_category_inflows_clean.csv',
 '06_industry_folio_count_clean.csv',
 '07_scheme_performance_clean.csv',
 '08_investor_transactions_clean.csv',
 '09_portfolio_holdings_clean.csv',
 '10_benchmark_indices_clean.csv']